In [ ]:
!pip install roboflow pycocotools scikit-image "transformers>=4.56" -q

In [ ]:
import os, cv2, json, random, contextlib

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from pycocotools.coco import COCO

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

from scipy.ndimage import distance_transform_edt, find_objects
from skimage.measure import label as sk_label
from skimage.segmentation import find_boundaries, watershed, relabel_sequential

from roboflow import Roboflow
from google.colab import userdata

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ROBOFLOW KEY (Colab secret)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

# HUGGING FACE TOKEN (Colab secret HF_TOKEN) — needed for the gated DINOv3
# weights. Without it (or before the licence is accepted on the model page)
# the notebook falls back to DINOv2 automatically.
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print('HF token found' if HF_TOKEN else 'No HF token → DINOv3 unavailable, DINOv2 will be used')

In [ ]:
!nvidia-smi

## DATASET PREPARATION

In [ ]:
# Access dataset from Roboflow (key comes from the ROBOFLOW_API_KEY Colab secret)
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("segmentation-iugrk").project("crop-field-bopay")
version = project.version(7)
dataset = version.download("coco-segmentation")

print("Dataset location:", dataset.location)

In [ ]:
base_dir = dataset.location  # your dataset root

for split in ['train', 'valid', 'test']:
    ann_file = os.path.join(base_dir, split, '_annotations.coco.json')

    if not os.path.exists(ann_file):
        continue

    print(f"\nFixing {split}...")

    with open(ann_file) as f:
        data = json.load(f)

    # Force SINGLE category
    data['categories'] = [{
        "id": 1,
        "name": "field",
        "supercategory": "field"
    }]
    for ann in data['annotations']:
        ann['category_id'] = 1

    with open(ann_file, "w") as f:
        json.dump(data, f)

    print(f"  ✓ {split} fixed")

print("\n✅ Dataset fully cleaned!")

In [ ]:
# Roboflow coco-segmentation layout:
#   <dataset.location>/
#     train/  _annotations.coco.json  *.jpg
#     valid/  _annotations.coco.json  *.jpg
#     test/   _annotations.coco.json  *.jpg

for split in ['train', 'valid', 'test']:
    split_dir = os.path.join(dataset.location, split)
    ann_file  = os.path.join(split_dir, '_annotations.coco.json')
    if os.path.exists(ann_file):
        with open(ann_file) as f:
            d = json.load(f)
        sizes = sorted({(im['width'], im['height']) for im in d['images']})
        print(f"[{split}] images: {len(d['images'])}  "
              f"annotations: {len(d['annotations'])}  "
              f"sizes (w,h): {sizes[:5]}{' …' if len(sizes) > 5 else ''}")

In [ ]:
IGNORE_INDEX = 255


def make_boundary(inst: np.ndarray, width: int = 1) -> np.ndarray:
    """
    Boundary map from an INSTANCE mask (0 = background, 1..N = fields).

    find_boundaries(mode='thick') marks every pixel whose neighbour has a
    different label, so it catches both field↔background edges AND the edge
    shared by two touching fields. (The previous version built boundaries from
    the semantic mask, where every field has label 1, so shared edges between
    adjacent fields were never marked — the main cause of merged fields.)

    width=1 gives a ~2px line; each extra step dilates it by 1px per side.
    """
    bound = find_boundaries(inst, connectivity=1, mode='thick')
    if width > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        bound = cv2.dilate(bound.astype(np.uint8), k, iterations=width - 1) > 0
    return bound.astype(np.float32)


def make_distance(inst: np.ndarray, bound: np.ndarray) -> np.ndarray:
    """
    Per-field normalised distance to the field's own boundary: 0 on the edge,
    1 at the field centre, for every field regardless of its size.

    Normalising per field (not per image) gives every field — small or large —
    a clear peak, which is what the watershed markers are seeded from.
    """
    dist = np.zeros(inst.shape, dtype=np.float32)
    interior = (inst > 0) & (bound == 0)
    for lbl, slc in enumerate(find_objects(inst), start=1):
        if slc is None:
            continue
        m = (inst[slc] == lbl) & interior[slc]
        if not m.any():
            continue
        d = distance_transform_edt(np.pad(m, 1))[1:-1, 1:-1]
        dist[slc][m] = (d[m] / d.max()).astype(np.float32)
    return dist


class CocoFieldDataset(Dataset):
    """
    Roboflow coco-segmentation reader for multitask field delineation.

    Returns (image, seg, boundary, dist, valid, inst):
      seg      (H,W) long   0 = background, 1 = field, 255 = padding (ignored)
      boundary (H,W) float  1 on field edges (incl. edges shared by two fields)
      dist     (H,W) float  per-field normalised distance to the edge
      valid    (H,W) float  1 on real pixels, 0 on padding
      inst     (H,W) long   instance ids (used for field-level validation)

    Training samples are random crops at native resolution (± scale jitter),
    so the model sees the same ground sampling distance as tiled inference.
    Targets are built AFTER the geometric transforms, so the boundary width
    and distance values are exact for every sample.
    """
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, split_dir, img_size=512, train=False,
                 scale_range=(0.75, 1.25), bound_width=1, max_eval_size=1024):
        ann_file = os.path.join(split_dir, '_annotations.coco.json')
        self.coco          = COCO(ann_file)
        self.img_ids       = list(self.coco.imgs.keys())
        self.split_dir     = split_dir
        self.img_size      = img_size
        self.train         = train
        self.scale_range   = scale_range
        self.bound_width   = bound_width
        self.max_eval_size = max_eval_size
        self.num_classes   = 2   # background + field
        self.jitter = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05)
        print(f'  {len(self.img_ids)} images | train={train}')

    def __len__(self):
        return len(self.img_ids)

    def _instance_mask(self, img_id, h, w):
        inst = np.zeros((h, w), dtype=np.int32)
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id, iscrowd=None))
        for i, ann in enumerate(anns, start=1):
            inst[self.coco.annToMask(ann) == 1] = i
        return inst

    @staticmethod
    def _crop_or_pad(img, inst, size, random_crop):
        h, w = inst.shape
        ch, cw = min(size, h), min(size, w)
        y0 = random.randint(0, h - ch) if random_crop else (h - ch) // 2
        x0 = random.randint(0, w - cw) if random_crop else (w - cw) // 2
        return img[y0:y0 + ch, x0:x0 + cw], inst[y0:y0 + ch, x0:x0 + cw]

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info   = self.coco.imgs[img_id]
        img    = np.array(Image.open(os.path.join(self.split_dir, info['file_name'])).convert('RGB'))
        inst   = self._instance_mask(img_id, img.shape[0], img.shape[1])

        if self.train:
            # ── Scale jitter around native resolution ─────────────────────────
            s = random.uniform(*self.scale_range)
            if abs(s - 1.0) > 1e-3:
                nh, nw = max(int(round(img.shape[0] * s)), 32), max(int(round(img.shape[1] * s)), 32)
                img  = cv2.resize(img,  (nw, nh), interpolation=cv2.INTER_LINEAR)
                inst = cv2.resize(inst, (nw, nh), interpolation=cv2.INTER_NEAREST)
            img, inst = self._crop_or_pad(img, inst, self.img_size, random_crop=True)
            # ── Dihedral augmentation (lossless: no interpolation, no corners)
            k = random.randint(0, 3)
            img, inst = np.rot90(img, k), np.rot90(inst, k)
            if random.random() < 0.5:
                img, inst = img[:, ::-1], inst[:, ::-1]
            out_h = out_w = self.img_size
        else:
            img, inst = self._crop_or_pad(img, inst, self.max_eval_size, random_crop=False)
            out_h = int(np.ceil(inst.shape[0] / 32) * 32)
            out_w = int(np.ceil(inst.shape[1] / 32) * 32)

        img, inst = np.ascontiguousarray(img), np.ascontiguousarray(inst)

        # ── Targets from the transformed instance mask ─────────────────────────
        bound = make_boundary(inst, self.bound_width)
        dist  = make_distance(inst, bound)
        seg   = (inst > 0).astype(np.int64)
        valid = np.ones(inst.shape, dtype=np.float32)

        # ── Pad to the output size (padding is ignored by every loss) ─────────
        ph, pw = out_h - inst.shape[0], out_w - inst.shape[1]
        if ph > 0 or pw > 0:
            pad2 = ((0, ph), (0, pw))
            img   = np.pad(img, pad2 + ((0, 0),))
            inst  = np.pad(inst,  pad2)
            bound = np.pad(bound, pad2)
            dist  = np.pad(dist,  pad2)
            valid = np.pad(valid, pad2)
            seg   = np.pad(seg,   pad2, constant_values=IGNORE_INDEX)

        img = TF.to_tensor(img)
        if self.train:
            if random.random() < 0.8:
                img = self.jitter(img)
            if random.random() < 0.3:
                img = TF.gaussian_blur(img, kernel_size=5, sigma=random.uniform(0.3, 1.5))
        img = TF.normalize(img, mean=self.MEAN, std=self.STD)

        return (img,
                torch.from_numpy(seg),
                torch.from_numpy(bound),
                torch.from_numpy(dist),
                torch.from_numpy(valid),
                torch.from_numpy(inst.astype(np.int64)))

# MODEL TRAINING

In [ ]:
IMG_SIZE = 512   # training crop size = inference tile size
BATCH    = 8
WORKERS  = 4

train_ds = CocoFieldDataset(os.path.join(dataset.location, 'train'), IMG_SIZE, train=True)
val_ds   = CocoFieldDataset(os.path.join(dataset.location, 'valid'), IMG_SIZE, train=False)
test_ds  = CocoFieldDataset(os.path.join(dataset.location, 'test'),  IMG_SIZE, train=False)

NUM_CLASSES = train_ds.num_classes

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True,
                          num_workers=WORKERS, pin_memory=True, persistent_workers=True)
# Validation/test use whole images (padded to a multiple of 32), one at a time
val_loader   = DataLoader(val_ds,  batch_size=1, shuffle=False, num_workers=WORKERS)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=WORKERS)
print(f'Train batches: {len(train_loader)} | Val images: {len(val_ds)} | Test images: {len(test_ds)}')

# Quick look at the targets: every field (incl. touching ones) must be outlined
img, seg, bound, dist, valid, inst = train_ds[0]
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow((img.permute(1, 2, 0) * torch.tensor(CocoFieldDataset.STD) + torch.tensor(CocoFieldDataset.MEAN)).clamp(0, 1))
ax[1].imshow(inst, cmap='tab20'); ax[2].imshow(bound, cmap='gray'); ax[3].imshow(dist, cmap='magma')
for a, t in zip(ax, ['Image', 'Instances', 'Boundary target', 'Distance target']):
    a.set_title(t); a.axis('off')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# DINO-FracTAL ResUNet
#
#   Image ──┬──► DINOv3 / DINOv2 ViT (frozen) ──► 32×32 token maps from
#           │                                    three depths (½, ¾, last)
#           │                                        │      │      │
#           └──► FracTAL encoder                     │      │      │
#                  enc1 → 256×256 @ 32ch  (s1)       │      │      │
#                  enc2 → 128×128 @ 64ch  (s2)       │      │      │
#                  enc3 →  64×64  @ 128ch (s3) ◄─────┘      │      │  GatedFusion
#                  enc4 →  32×32  @ 256ch (s4) ◄────────────┘      │  GatedFusion
#                  bottleneck → 32×32 @ 512ch  ◄───────────────────┘  GatedFusion
#                  decoder → 512×512 → seg | boundary | distance heads
#
# Backbone (first one that loads):
#   1. DINOv3 ViT-L/16 SAT-493M  — pretrained on satellite imagery (gated)
#   2. DINOv3 ViT-B/16 LVD-1689M — pretrained on web images (gated)
#   3. DINOv2 ViT-B/14 + registers (Apache-2.0, not gated)
# DINO's dense features are strongest at telling land-cover types apart
# (field vs. orchard, tree line, road, fallow). The FracTAL decoder with its
# full-resolution skips still draws the precise edges.
# ════════════════════════════════════════════════════════════════════════════

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def amp_dtype(device):
    """bf16 on GPUs that support it (A100, L4…), fp16 otherwise (T4), None on CPU."""
    if device.type != 'cuda':
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def autocast_ctx(device):
    dtype = amp_dtype(device)
    return torch.autocast('cuda', dtype=dtype) if dtype else contextlib.nullcontext()


class GatedFusion(nn.Module):
    """
    Spatially aligned fusion of foundation-model features into a U-Net level.

        e   = refine(proj(foundation_feat))       resized to the level's grid
        g   = sigmoid(conv1x1([x, e]))            per-pixel, per-channel gate
        out = x + gamma * g * e

    Why this replaces the old cross-attention block:
      • The foundation features are already on the same spatial grid as the
        U-Net features, so each position simply reads its own location. The
        old attention had no positional encoding, so every position attended
        to an unordered bag of SAM tokens and lost where things were.
      • gamma starts at 0.5 (not 0) and trains at the normal learning rate
        without weight decay, so the fusion can actually open. The old gate
        was capped at ~0.03 by its 1e-5 LR and ended at 0.012.
    """
    def __init__(self, ch: int, ext_ch: int, init_gamma: float = 0.5):
        super().__init__()
        self.proj   = nn.Sequential(nn.Conv2d(ext_ch, ch, 1, bias=False),
                                    nn.BatchNorm2d(ch), nn.GELU())
        self.refine = nn.Sequential(nn.Conv2d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm2d(ch))
        self.gate   = nn.Sequential(nn.Conv2d(2 * ch, ch, 1), nn.Sigmoid())
        self.gamma  = nn.Parameter(torch.tensor(float(init_gamma)))

    def forward(self, x: torch.Tensor, ext: torch.Tensor) -> torch.Tensor:
        e = self.proj(ext.to(x.dtype))
        if e.shape[-2:] != x.shape[-2:]:
            e = F.interpolate(e, size=x.shape[-2:], mode='bilinear', align_corners=False)
        e = self.refine(e)
        g = self.gate(torch.cat([x, e], dim=1))
        return x + self.gamma * g * e


# ── FracTAL ResUNet components (unchanged) ───────────────────────────────────

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, max(channels//reduction, 4)), nn.ReLU(),
            nn.Linear(max(channels//reduction, 4), channels), nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(x).view(x.size(0), x.size(1), 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 1)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(x))


class FracTALUnit(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ch_att = ChannelAttention(channels)
        self.sp_att = SpatialAttention(channels)
        self.alpha  = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        a = torch.clamp(self.alpha, 0.0, 1.0)
        return a * self.ch_att(x) + (1-a) * self.sp_att(x)


class FracTALResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=dilation, dilation=dilation, bias=False),
        )
        self.fractal = FracTALUnit(out_ch)
        self.proj    = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.proj(x) + self.fractal(self.conv(x))


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_blocks=2, dilation=1):
        super().__init__()
        blocks = [FracTALResBlock(in_ch, out_ch, dilation)]
        for _ in range(n_blocks-1):
            blocks.append(FracTALResBlock(out_ch, out_ch, dilation))
        self.blocks = nn.Sequential(*blocks)
        self.pool   = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.blocks(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.block = FracTALResBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.block(torch.cat([x, skip], dim=1))


# ── FracTAL ResUNet + frozen foundation encoder ──────────────────────────────

class FoundationFracTALResUNet(nn.Module):
    """
    FracTAL ResUNet with a frozen foundation encoder fused at three levels:

        level   resolution (512 input)   U-Net features fused
        's3'    64×64                    enc3 skip  (fine structure, edges)
        's4'    32×32                    enc4 skip
        'bott'  32×32                    bottleneck (field extent, semantics)

    `foundation` is any module returning a list of feature maps and exposing
    `out_channels` (one entry per map). `fusion_sources` says which of those
    maps feeds each level, e.g. {'s3': 0, 's4': 1, 'bott': 2}.
    With foundation=None the model is the plain FracTAL ResUNet baseline.
    """
    def __init__(self, foundation=None, fusion_sources=None,
                 num_classes: int = 2, base_ch: int = 32):
        super().__init__()
        c = base_ch
        self.foundation = foundation
        self.fusion_sources = dict(fusion_sources or {})

        self.enc1       = EncoderBlock(3,   c,   n_blocks=2)
        self.enc2       = EncoderBlock(c,   c*2, n_blocks=2)
        self.enc3       = EncoderBlock(c*2, c*4, n_blocks=3)
        self.enc4       = EncoderBlock(c*4, c*8, n_blocks=3)
        self.bottleneck = nn.Sequential(
            FracTALResBlock(c*8,  c*16, dilation=2),
            FracTALResBlock(c*16, c*16, dilation=4),
            FracTALResBlock(c*16, c*16, dilation=2),
        )

        level_ch = {'s3': c*4, 's4': c*8, 'bott': c*16}
        self.fusions = nn.ModuleDict()
        if foundation is not None:
            for level, src in self.fusion_sources.items():
                self.fusions[level] = GatedFusion(level_ch[level], foundation.out_channels[src])

        self.dec4 = DecoderBlock(c*16, c*8, c*8)
        self.dec3 = DecoderBlock(c*8,  c*4, c*4)
        self.dec2 = DecoderBlock(c*4,  c*2, c*2)
        self.dec1 = DecoderBlock(c*2,  c,   c)

        self.seg_head   = nn.Conv2d(c, num_classes, 1)
        self.bound_head = nn.Conv2d(c, 1, 1)
        self.dist_head  = nn.Conv2d(c, 1, 1)

    def _fuse(self, level, x, feats):
        if level in self.fusions:
            return self.fusions[level](x, feats[self.fusion_sources[level]])
        return x

    def forward(self, x: torch.Tensor) -> dict:
        feats = self.foundation(x) if self.foundation is not None else None

        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x     = self.bottleneck(x)

        s3 = self._fuse('s3',   s3, feats)
        s4 = self._fuse('s4',   s4, feats)
        x  = self._fuse('bott', x,  feats)

        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)

        return {'seg':   self.seg_head(x),
                'bound': self.bound_head(x),
                'dist':  self.dist_head(x)}

    def train(self, mode: bool = True):
        super().train(mode)
        if self.foundation is not None:
            self.foundation.eval()   # frozen: keep its norm/dropout layers in eval mode
        return self

    def fusion_strengths(self) -> dict:
        return {k: float(f.gamma.detach()) for k, f in self.fusions.items()}

    def trainable_state_dict(self) -> dict:
        """Everything except the frozen foundation weights (loaded separately)."""
        return {k: v for k, v in self.state_dict().items()
                if not k.startswith('foundation.')}

    def load_trainable_state_dict(self, state_dict):
        missing, unexpected = self.load_state_dict(state_dict, strict=False)
        missing = [k for k in missing if not k.startswith('foundation.')]
        if missing or unexpected:
            raise RuntimeError(f'Checkpoint mismatch. missing={missing[:5]} unexpected={unexpected[:5]}')
        return self


# Backbones tried in order; the first one that loads is used.
#   DINOv3 weights are free to use under Meta's DINOv3 License but GATED on
#   Hugging Face: accept the licence on the model page once, then add an
#   HF_TOKEN secret in Colab. If that isn't done, loading falls back to DINOv2
#   (Apache-2.0, not gated).
DINO_CANDIDATES = [
    'facebook/dinov3-vitl16-pretrain-sat493m',    # DINOv3 ViT-L/16, satellite-pretrained (SAT-493M)
    'facebook/dinov3-vitb16-pretrain-lvd1689m',   # DINOv3 ViT-B/16, web-image pretrained
    'facebook/dinov2-with-registers-base',        # DINOv2 ViT-B/14 + registers (fallback)
]

# Normalisation published with the DINOv3 satellite models; everything else
# uses ImageNet statistics.
SAT_MEAN = (0.430, 0.411, 0.296)
SAT_STD  = (0.213, 0.156, 0.143)


class DINOEncoder(nn.Module):
    """
    Frozen DINOv3 / DINOv2 ViT returning patch-token maps from three depths.

    Input : (B,3,H,W) ImageNet-normalised, H and W multiples of 16
    Output: [shallow, middle, deep] each (B, C, H/16, W/16)

    The tile is resized so the token grid is exactly H/16 × W/16 whatever the
    patch size (16 for DINOv3, 14 for DINOv2), keeping it aligned with the
    U-Net's 1/16 bottleneck. Three depths feed three U-Net levels: shallower
    layers carry more local/edge detail, the last layer the most semantics.
    """
    def __init__(self, model_name: str, layer_fractions=(0.5, 0.75, 1.0), token=None):
        super().__init__()
        from transformers import AutoModel
        self.model_name = model_name
        self.backbone   = AutoModel.from_pretrained(model_name, token=token)
        self.backbone.requires_grad_(False)
        self.backbone.eval()

        cfg = self.backbone.config
        self.patch   = cfg.patch_size
        n_layers     = cfg.num_hidden_layers
        # hidden_states[0] is the patch embedding; [i] is the output of block i
        self.layers  = [max(1, round(f * n_layers)) for f in layer_fractions]
        self.out_channels = [cfg.hidden_size] * len(self.layers)

        mean, std = (SAT_MEAN, SAT_STD) if 'sat493m' in model_name else (IMAGENET_MEAN, IMAGENET_STD)
        self.register_buffer('mean',    torch.tensor(mean).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('std',     torch.tensor(std).view(1, 3, 1, 1),  persistent=False)
        self.register_buffer('in_mean', torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer('in_std',  torch.tensor(IMAGENET_STD).view(1, 3, 1, 1),  persistent=False)

    @torch.no_grad()
    def forward(self, x: torch.Tensor):
        B, _, H, W = x.shape
        gh, gw = H // 16, W // 16
        x = (x * self.in_std + self.in_mean - self.mean) / self.std
        size = (gh * self.patch, gw * self.patch)
        if x.shape[-2:] != size:
            x = F.interpolate(x, size=size, mode='bilinear', align_corners=False)

        hs = self.backbone(pixel_values=x, output_hidden_states=True).hidden_states
        feats = []
        for i in self.layers:
            t = hs[i]
            t = t[:, t.shape[1] - gh * gw:, :]          # drop CLS + register tokens
            t = F.layer_norm(t, t.shape[-1:])
            feats.append(t.transpose(1, 2).reshape(B, -1, gh, gw))
        return feats


def load_dino_encoder(candidates=DINO_CANDIDATES, token=None):
    last_err = None
    for name in candidates:
        try:
            enc = DINOEncoder(name, token=token)
            print(f'✓ Loaded {name}  (patch={enc.patch}, layers={enc.layers}, '
                  f'dim={enc.out_channels[0]})')
            return enc
        except Exception as e:  # gated / not accepted / offline
            print(f'✗ {name}: {type(e).__name__}: {str(e).splitlines()[0][:120]}')
            last_err = e
    raise RuntimeError('No DINO backbone could be loaded') from last_err

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/AGRI/CropField_Segmentation/result/dino-fractal-resunet-512-v01'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

encoder = load_dino_encoder(DINO_CANDIDATES, token=HF_TOKEN)

MODEL_CONFIG = {
    'encoder':        'dino',
    'backbone_name':  encoder.model_name,
    'fusion_sources': {'s3': 0, 's4': 1, 'bott': 2},   # shallow → fine level, deep → bottleneck
    'num_classes':    NUM_CLASSES,
    'base_ch':        32,
    'img_size':       IMG_SIZE,
}

model = FoundationFracTALResUNet(
    foundation     = encoder,
    fusion_sources = MODEL_CONFIG['fusion_sources'],
    num_classes    = NUM_CLASSES,
    base_ch        = MODEL_CONFIG['base_ch'],
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Backbone         : {encoder.model_name}')
print(f'Trainable params : {trainable:,}   ← FracTAL + fusion')
print(f'Frozen params    : {frozen:,}   ← DINO ViT')
print(f'Fusion strengths : {model.fusion_strengths()}')

with torch.no_grad(), autocast_ctx(DEVICE):
    out = model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
for k, v in out.items():
    print(f'  {k}: {tuple(v.shape)}')

In [ ]:
POSTPROCESS_DEFAULTS = {
    'extent_thresh' : 0.50,  # field probability above which a pixel belongs to a field
    'marker_dist'   : 0.40,  # distance-head value that marks a field's core
    'marker_bound'  : 0.30,  # ...only where the boundary head is below this
    'min_marker_px' : 20,    # ignore tiny seeds (noise)
    'min_field_px'  : 100,   # drop fields smaller than this
}


def delineate_fields(field_p: np.ndarray, bound_p: np.ndarray, dist_p: np.ndarray,
                     cfg: dict = POSTPROCESS_DEFAULTS) -> np.ndarray:
    """
    Probability maps → instance label map (0 = background, 1..N = fields).

    Seeded watershed, as in the original FracTAL ResUNet paper:
      1. extent  = pixels the segmentation head calls field
      2. markers = field cores: high distance-to-edge, low boundary probability
      3. flood the energy (boundary − distance) from the markers inside the
         extent, so touching fields split along the predicted boundary while
         each keeps its full extent (no boundary strip is cut away, so no
         buffering is needed afterwards)
      4. extent pieces that received no marker are kept as their own fields
    """
    extent  = field_p > cfg['extent_thresh']
    cores   = (dist_p > cfg['marker_dist']) & (bound_p < cfg['marker_bound']) & extent
    markers = _drop_small(sk_label(cores, connectivity=1), cfg['min_marker_px'])

    energy  = bound_p.astype(np.float32) - dist_p.astype(np.float32)
    labels  = watershed(energy, markers, mask=extent)

    orphans = sk_label(extent & (labels == 0), connectivity=1)
    if orphans.max() > 0:
        labels = np.where(orphans > 0, orphans + labels.max(), labels)

    return _drop_small(labels, cfg['min_field_px']).astype(np.int32)


def _drop_small(labels: np.ndarray, min_px: int) -> np.ndarray:
    """Remove labelled regions smaller than min_px and relabel 1..N."""
    small = np.bincount(labels.ravel()) < min_px
    small[0] = False
    labels = np.where(small[labels], 0, labels)
    return relabel_sequential(labels)[0]


def match_fields(pred: np.ndarray, gt: np.ndarray, iou_thresh: float = 0.5) -> dict:
    """
    Field-level (object) matching between two instance maps.
    With IoU > 0.5 each field can match at most one field on the other side.

    Returns counts (tp, fp, fn) plus the summed IoU of matched pairs, so they
    can be pooled over a whole validation set before computing F1.
    """
    pred, _, _ = relabel_sequential(pred.astype(np.int64))
    gt,   _, _ = relabel_sequential(gt.astype(np.int64))
    n_p, n_g = int(pred.max()), int(gt.max())
    if n_p == 0 or n_g == 0:
        return {'tp': 0, 'fp': n_p, 'fn': n_g, 'iou_sum': 0.0}
    ov = np.bincount(pred.ravel() * (n_g + 1) + gt.ravel(),
                     minlength=(n_p + 1) * (n_g + 1)).reshape(n_p + 1, n_g + 1)
    inter = ov[1:, 1:].astype(np.float64)
    union = ov[1:, :].sum(1)[:, None] + ov[:, 1:].sum(0)[None, :] - inter
    iou   = inter / np.maximum(union, 1)
    hit   = iou > iou_thresh
    tp    = int(hit.sum())
    return {'tp': tp, 'fp': n_p - tp, 'fn': n_g - tp, 'iou_sum': float(iou[hit].sum())}


def summarise_matches(m: dict) -> dict:
    tp, fp, fn = m['tp'], m['fp'], m['fn']
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    return {'field_f1': 2 * prec * rec / max(prec + rec, 1e-9),
            'field_precision': prec, 'field_recall': rec,
            'matched_iou': m['iou_sum'] / max(tp, 1)}


# ════════════════════════════════════════════════════════════════════════════
# Losses and metrics
#
# Changes vs. the previous version (which produced ~46% boundary coverage):
#   • boundary BCE pos_weight 20 → 3, plus a soft-Dice term: the Dice term
#     rewards overlap with the thin target line instead of recall at any cost
#   • boundary weight inside the segmentation CE 12× → 3×
#   • distance loss averaged over field pixels only (was diluted by all pixels)
#   • padding pixels (valid = 0) are ignored everywhere
#   • boundary F1 uses a 2px tolerance and is pooled over the whole epoch
# ════════════════════════════════════════════════════════════════════════════

LAMBDA_DICE      = 1.0
LAMBDA_BOUND     = 1.0
LAMBDA_DIST      = 1.0
BOUND_CE_WEIGHT  = 3.0
BOUND_POS_WEIGHT = 3.0
BF1_TOLERANCE    = 2     # px


def soft_dice(prob, target, valid, eps=1.0):
    inter = (prob * target * valid).sum(dim=(1, 2))
    denom = ((prob + target) * valid).sum(dim=(1, 2))
    return (1.0 - (2.0 * inter + eps) / (denom + eps)).mean()


def fractal_loss(out, seg, bound, dist, valid):
    n_valid = valid.sum().clamp(min=1.0)

    # Segmentation: boundary-weighted CE + soft Dice on the field class
    ce     = F.cross_entropy(out['seg'], seg, ignore_index=IGNORE_INDEX, reduction='none')
    l_ce   = (ce * (1.0 + BOUND_CE_WEIGHT * bound) * valid).sum() / n_valid
    p_fld  = torch.softmax(out['seg'], dim=1)[:, 1]
    l_dice = soft_dice(p_fld, (seg == 1).float(), valid)

    # Boundary: mildly weighted BCE + soft Dice
    b_logit = out['bound'].squeeze(1)
    bce     = F.binary_cross_entropy_with_logits(
        b_logit, bound, reduction='none',
        pos_weight=torch.tensor(BOUND_POS_WEIGHT, device=bound.device))
    l_bound = (bce * valid).sum() / n_valid + soft_dice(torch.sigmoid(b_logit), bound, valid)

    # Distance: MSE on field pixels only
    fld    = (seg == 1).float() * valid
    d_pred = torch.sigmoid(out['dist'].squeeze(1))
    l_dist = ((d_pred - dist) ** 2 * fld).sum() / fld.sum().clamp(min=1.0)

    total = l_ce + LAMBDA_DICE * l_dice + LAMBDA_BOUND * l_bound + LAMBDA_DIST * l_dist
    return total, {'ce': l_ce.item(), 'dice': l_dice.item(),
                   'bound': l_bound.item(), 'dist': l_dist.item()}


@torch.no_grad()
def pixel_counts(out, seg, bound, valid, tol=BF1_TOLERANCE):
    """Counts for field IoU and tolerance-based boundary F1 (pooled later)."""
    v     = valid > 0
    pred  = (out['seg'].argmax(1) == 1) & v
    true  = (seg == 1) & v
    pb    = (torch.sigmoid(out['bound'].squeeze(1).float()) > 0.5) & v
    tb    = (bound > 0.5) & v
    k     = 2 * tol + 1
    dil   = lambda m: F.max_pool2d(m.float().unsqueeze(1), k, 1, tol).squeeze(1) > 0
    return {'inter':   (pred & true).sum().item(),
            'union':   (pred | true).sum().item(),
            'b_pred':  pb.sum().item(),
            'b_true':  tb.sum().item(),
            'b_prec_hit': (pb & dil(tb)).sum().item(),
            'b_rec_hit':  (tb & dil(pb)).sum().item()}


def summarise_pixels(c):
    prec = c['b_prec_hit'] / max(c['b_pred'], 1)
    rec  = c['b_rec_hit']  / max(c['b_true'], 1)
    return {'field_iou': c['inter'] / max(c['union'], 1),
            'bound_f1':  2 * prec * rec / max(prec + rec, 1e-9)}


def train_one_epoch(model, loader, optimizer, device, scaler):
    model.train()
    tot, parts, n = 0.0, {}, 0
    counts = {}
    for img, seg, bound, dist, valid, _ in tqdm(loader, desc='train', leave=False):
        img, seg, bound, dist, valid = (t.to(device, non_blocking=True)
                                        for t in (img, seg, bound, dist, valid))
        with autocast_ctx(device):
            out = model(img)
        out = {k: v.float() for k, v in out.items()}
        loss, p = fractal_loss(out, seg, bound, dist, valid)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        tot += loss.item(); n += 1
        for k, v in p.items():
            parts[k] = parts.get(k, 0.0) + v
        for k, v in pixel_counts(out, seg, bound, valid).items():
            counts[k] = counts.get(k, 0) + v
    return {'loss': tot / n, **{k: v / n for k, v in parts.items()}, **summarise_pixels(counts)}


@torch.no_grad()
def predict_probs(model, img, device):
    """(1,3,H,W) → field, boundary, distance probability maps as numpy."""
    with autocast_ctx(device):
        out = model(img.to(device))
    out = {k: v.float() for k, v in out.items()}
    return (out,
            torch.softmax(out['seg'], 1)[0, 1].cpu().numpy(),
            torch.sigmoid(out['bound'])[0, 0].cpu().numpy(),
            torch.sigmoid(out['dist'])[0, 0].cpu().numpy())


@torch.no_grad()
def evaluate(model, loader, device, post_cfg=POSTPROCESS_DEFAULTS, keep_probs=False):
    """
    Pixel metrics + FIELD-LEVEL metrics: every validation image goes through
    the same watershed post-processing as inference and its fields are matched
    to the ground-truth fields (IoU > 0.5).
    """
    model.eval()
    tot, parts, n = 0.0, {}, 0
    counts, fm = {}, {'tp': 0, 'fp': 0, 'fn': 0, 'iou_sum': 0.0}
    cache = []
    for img, seg, bound, dist, valid, inst in tqdm(loader, desc='val  ', leave=False):
        seg, bound, dist, valid = (t.to(device) for t in (seg, bound, dist, valid))
        out, fp_, bp_, dp_ = predict_probs(model, img, device)
        loss, p = fractal_loss(out, seg, bound, dist, valid)
        tot += loss.item(); n += 1
        for k, v in p.items():
            parts[k] = parts.get(k, 0.0) + v
        for k, v in pixel_counts(out, seg, bound, valid).items():
            counts[k] = counts.get(k, 0) + v

        v_np  = valid[0].cpu().numpy() > 0
        gt    = inst[0].numpy() * v_np
        pred  = delineate_fields(fp_ * v_np, bp_, dp_, post_cfg)
        for k, v in match_fields(pred, gt).items():
            fm[k] += v
        if keep_probs:
            cache.append((fp_ * v_np, bp_, dp_, gt))
    res = {'loss': tot / n, **{k: v / n for k, v in parts.items()},
           **summarise_pixels(counts), **summarise_matches(fm)}
    return (res, cache) if keep_probs else res

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Optimizer
#
# One learning rate for everything trainable. Biases, norm layers and the
# fusion strengths (gamma) are excluded from weight decay, so decay no longer
# drags the fusion back towards 0.
# ════════════════════════════════════════════════════════════════════════════

EPOCHS   = 150
LR       = 2e-4
WARMUP   = 5
PATIENCE = 30     # stop when val field-F1 hasn't improved for this many epochs

decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    (no_decay if p.ndim <= 1 else decay).append(p)

optimizer = optim.AdamW([{'params': decay,    'weight_decay': 1e-4},
                         {'params': no_decay, 'weight_decay': 0.0}], lr=LR)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    progress = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler('cuda', enabled=amp_dtype(DEVICE) == torch.float16)
print(f'AMP dtype: {amp_dtype(DEVICE)}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Training
#
# Model selection uses FIELD-LEVEL F1 on the validation set: each val image is
# run through the same watershed post-processing as inference and its fields
# are matched to the ground-truth fields (IoU > 0.5). This measures what the
# pipeline actually delivers (separate, complete fields), not pixel overlap.
# ════════════════════════════════════════════════════════════════════════════

os.makedirs(OUTPUT_DIR, exist_ok=True)
history = {}
best_score, since_best = -1.0, 0

def log(prefix, d):
    for k, v in d.items():
        history.setdefault(f'{prefix}_{k}', []).append(v)

for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch(model, train_loader, optimizer, DEVICE, scaler)
    vl = evaluate(model, val_loader, DEVICE)
    scheduler.step()

    log('train', tr); log('val', vl)
    fusion = model.fusion_strengths()
    history.setdefault('fusion', []).append(fusion)

    print(f"Epoch {epoch:3d}/{EPOCHS} | LR={optimizer.param_groups[0]['lr']:.2e} | "
          f"Loss {tr['loss']:.3f}/{vl['loss']:.3f} | "
          f"IoU {tr['field_iou']:.3f}/{vl['field_iou']:.3f} | "
          f"BndF1@{BF1_TOLERANCE}px {tr['bound_f1']:.3f}/{vl['bound_f1']:.3f} | "
          f"FieldF1 {vl['field_f1']:.3f} (P {vl['field_precision']:.2f} R {vl['field_recall']:.2f}) | "
          f"fusion γ " + ' '.join(f'{k}={v:.2f}' for k, v in fusion.items()))

    score = vl['field_f1'] + 1e-3 * vl['bound_f1']   # boundary F1 breaks ties
    ckpt = {'epoch': epoch, 'state_dict': model.trainable_state_dict(),
            'model_config': MODEL_CONFIG, 'postprocess': dict(POSTPROCESS_DEFAULTS),
            'val_metrics': vl}
    if score > best_score:
        best_score, since_best = score, 0
        torch.save(ckpt, os.path.join(OUTPUT_DIR, 'best.pt'))
        print(f"  ✓ New best val FieldF1: {vl['field_f1']:.4f}  (IoU={vl['field_iou']:.4f}, "
              f"BndF1={vl['bound_f1']:.4f})")
    else:
        since_best += 1

    torch.save({**ckpt, 'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(), 'history': history},
               os.path.join(OUTPUT_DIR, 'last.pt'))

    if since_best >= PATIENCE:
        print(f'Early stopping: no improvement for {PATIENCE} epochs')
        break

print(f'\nTraining complete. Best val FieldF1: {best_score:.4f}')

## VISUALIZATION

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

axes[0,0].plot(history['train_loss'], label='Train')
axes[0,0].plot(history['val_loss'],   label='Val')
axes[0,0].set_title('Total loss'); axes[0,0].legend()

axes[0,1].plot(history['train_field_iou'], label='Train')
axes[0,1].plot(history['val_field_iou'],   label='Val')
axes[0,1].plot(history['val_bound_f1'],    label=f'Val boundary F1 @{BF1_TOLERANCE}px')
axes[0,1].set_title('Pixel metrics'); axes[0,1].legend()

axes[1,0].plot(history['val_field_f1'],        label='Field F1  ← model selection')
axes[1,0].plot(history['val_field_precision'], label='Field precision', alpha=0.6)
axes[1,0].plot(history['val_field_recall'],    label='Field recall',    alpha=0.6)
axes[1,0].plot(history['val_matched_iou'],     label='IoU of matched fields', alpha=0.6)
axes[1,0].set_title('Field-level metrics (val, after watershed)'); axes[1,0].legend()

for level in history['fusion'][0]:
    axes[1,1].plot([f[level] for f in history['fusion']], label=level)
axes[1,1].axhline(0, color='gray', linestyle=':', linewidth=1)
axes[1,1].set_title('Fusion strength γ per level (0 = foundation features unused)')
axes[1,1].legend()

for ax in axes.flat: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## POST-PROCESSING TUNING & TEST EVALUATION

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. Load the best checkpoint
# 2. Grid-search the watershed thresholds on the VALIDATION set
# 3. Store the winning thresholds in best.pt (the inference notebook uses them)
# 4. Report final metrics on the untouched TEST set
# ════════════════════════════════════════════════════════════════════════════
import itertools

best_path = os.path.join(OUTPUT_DIR, 'best.pt')
ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
model.load_trainable_state_dict(ckpt['state_dict'])
model.eval()

_, val_cache = evaluate(model, val_loader, DEVICE, keep_probs=True)

GRID = {'extent_thresh': [0.4, 0.5, 0.6],
        'marker_dist':   [0.3, 0.4, 0.5, 0.6],
        'marker_bound':  [0.2, 0.3, 0.5]}

results = []
for values in itertools.product(*GRID.values()):
    cfg = {**POSTPROCESS_DEFAULTS, **dict(zip(GRID, values))}
    fm  = {'tp': 0, 'fp': 0, 'fn': 0, 'iou_sum': 0.0}
    for field_p, bound_p, dist_p, gt in val_cache:
        for k, v in match_fields(delineate_fields(field_p, bound_p, dist_p, cfg), gt).items():
            fm[k] += v
    results.append((summarise_matches(fm)['field_f1'], cfg))

results.sort(key=lambda r: r[0], reverse=True)
best_f1, best_cfg = results[0]
print('Top settings on validation:')
for f1, cfg in results[:5]:
    print(f'  FieldF1={f1:.4f}  ' + '  '.join(f'{k}={cfg[k]}' for k in GRID))

ckpt['postprocess'] = best_cfg
torch.save(ckpt, best_path)
print(f'\nSaved tuned post-processing to {best_path}')

test_res = evaluate(model, test_loader, DEVICE, post_cfg=best_cfg)
print('\nTEST SET (best checkpoint, tuned post-processing)')
for k in ['field_f1', 'field_precision', 'field_recall', 'matched_iou', 'field_iou', 'bound_f1']:
    print(f'  {k:16s}: {test_res[k]:.4f}')

In [ ]:
# Visual check on validation images: probabilities → watershed fields
n_show = min(3, len(val_cache))
fig, axes = plt.subplots(n_show, 5, figsize=(25, 5 * n_show), squeeze=False)
rng = np.random.default_rng(0)
for r in range(n_show):
    field_p, bound_p, dist_p, gt = val_cache[r]
    pred = delineate_fields(field_p, bound_p, dist_p, best_cfg)
    img  = val_ds[r][0].permute(1, 2, 0).numpy() * CocoFieldDataset.STD + CocoFieldDataset.MEAN
    img  = np.clip(img[:pred.shape[0], :pred.shape[1]], 0, 1)
    lut  = rng.uniform(0.2, 1.0, (max(pred.max(), gt.max()) + 1, 3)); lut[0] = 0
    for a, (im, t) in zip(axes[r], [(img, 'Image'), (field_p, 'Field prob'),
                                   (bound_p, 'Boundary prob'), (lut[gt], f'Ground truth ({gt.max()} fields)'),
                                   (lut[pred], f'Predicted ({pred.max()} fields)')]):
        a.imshow(im, cmap='magma' if im.ndim == 2 else None); a.set_title(t); a.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'val_predictions.png'), dpi=120, bbox_inches='tight')
plt.show()